# Priority Prediction Model

Predict whether an event will be High or Low priority using event attributes.

In [15]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [16]:
events = pd.read_csv(
    "../data/Astram event data anonymized.csv"
)

events.shape

(8173, 46)

In [17]:
events["start_datetime"] = pd.to_datetime(
    events["start_datetime"],
    errors="coerce"
)

events["year"] = events["start_datetime"].dt.year
events["month"] = events["start_datetime"].dt.month
events["day"] = events["start_datetime"].dt.day
events["hour"] = events["start_datetime"].dt.hour
events["weekday"] = events["start_datetime"].dt.day_name()

In [18]:
events = events.dropna(subset=["priority"])

events["priority"] = events["priority"].map({
    "Low":0,
    "High":1
})

events["priority"].value_counts()

priority
1    5030
0    3141
Name: count, dtype: int64

In [29]:
features = [
    "event_type",
    "event_cause",
    "zone",
    "requires_road_closure",
    "hour",
    "weekday",
    "zone_risk",
    "cause_risk",
    "hour_risk"
]

data = events[features + ["priority"]].copy()

data.head()

,event_type,event_cause,zone,requires_road_closure,hour,weekday,zone_risk,cause_risk,hour_risk,priority
0,unplanned,vehicle_breakdown,NaN,False,17.0,Thursday,NaN,0.662239,0.617647,1
1,unplanned,vehicle_breakdown,NaN,False,4.0,Tuesday,NaN,0.662239,0.655914,1
2,unplanned,others,Central Zone 2,False,6.0,Saturday,0.537721,0.595611,0.546970,0
3,unplanned,tree_fall,NaN,True,17.0,Thursday,NaN,0.327465,0.617647,0
4,unplanned,vehicle_breakdown,NaN,False,4.0,Tuesday,NaN,0.662239,0.655914,0


In [30]:
X = data[features]
y = data["priority"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
# Fill missing categorical values

for col in ["event_type", "event_cause", "zone", "weekday"]:
    X_train[col] = X_train[col].fillna("Unknown")
    X_valid[col] = X_valid[col].fillna("Unknown")

# Fill missing boolean/numeric values

X_train["requires_road_closure"] = (
    X_train["requires_road_closure"].fillna(False)
)

X_valid["requires_road_closure"] = (
    X_valid["requires_road_closure"].fillna(False)
)

X_train["hour"] = X_train["hour"].fillna(-1)
X_valid["hour"] = X_valid["hour"].fillna(-1)

In [32]:
cat_features = [
    "event_type",
    "event_cause",
    "zone",
    "weekday"
]

model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="Accuracy",
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 0.6533048	total: 73.9ms	remaining: 1m 13s
100:	learn: 0.6941554	total: 6.85s	remaining: 1m 1s
200:	learn: 0.7204712	total: 13.8s	remaining: 54.7s
300:	learn: 0.7463280	total: 22.5s	remaining: 52.2s
400:	learn: 0.7714198	total: 30s	remaining: 44.8s
500:	learn: 0.7868727	total: 37.3s	remaining: 37.1s
600:	learn: 0.8056916	total: 44.7s	remaining: 29.7s
700:	learn: 0.8190024	total: 51.6s	remaining: 22s
800:	learn: 0.8304774	total: 59.5s	remaining: 14.8s
900:	learn: 0.8414933	total: 1m 6s	remaining: 7.27s
999:	learn: 0.8532742	total: 1m 12s	remaining: 0us


CatBoostClassifier(depth=8, eval_metric='Accuracy', iterations=1000, learning_rate=0.05, loss_function='Logloss', verbose=100)

In [33]:
pred = model.predict(X_valid)

accuracy_score(
    y_valid,
    pred
)

0.6654434250764526

In [34]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_valid,
        pred
    )
)

              precision    recall  f1-score   support

           0       0.62      0.34      0.44       629
           1       0.68      0.87      0.76      1006

    accuracy                           0.67      1635
   macro avg       0.65      0.60      0.60      1635
weighted avg       0.66      0.67      0.64      1635



In [36]:
zone_risk = (
    events.groupby("zone")["priority"]
    .mean()
)

cause_risk = (
    events.groupby("event_cause")["priority"]
    .mean()
)

hour_risk = (
    events.groupby("hour")["priority"]
    .mean()
)

events["zone_risk"] = events["zone"].map(zone_risk)
events["cause_risk"] = events["event_cause"].map(cause_risk)
events["hour_risk"] = events["hour"].map(hour_risk)